##Dim User

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@azstoragespotify.dfs.core.windows.net/DimUser")

In [0]:
df.display()

df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys

project_pth = os.path.join(os.getcwd(),'..','..')
display(project_pth)
sys.path.insert(0, project_pth)
sys.path.append(project_pth)

from utils.transformations import reusable

In [0]:
dbutils.fs.rm("abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser", True)

#### auto Loader

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser/checkpoint")\
    .load("abfss://bronze@azstoragespotify.dfs.core.windows.net/DimUser")

In [0]:
display(df_user, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser/checkpoint")

In [0]:
df_user = df_user.withColumn('user_name',upper(col('user_name')))


In [0]:
df_user.printSchema()

In [0]:
print(df_user.isStreaming)

##### I created methods for transformations that I can reuse whenever I need to perform them

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])


In [0]:
display(df_user, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser/checkpoint")

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@azstoragespotify.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_catalog.silver.DimUserspotify_catalog.silver.dimuser")

##DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("schemaEvaluationMode","addNewColumn")\
    .load("abfss://bronze@azstoragespotify.dfs.core.windows.net/DimArtist")
    

In [0]:
display(df_art, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimArtist/checkpoint")

In [0]:
df_art = df_art.withColumn('artist_name',upper(col('artist_name')))

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
display(df_art, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimArtist/checkpoint")

In [0]:
df_art.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimArtist/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@azstoragespotify.dfs.core.windows.net/DimArtist/data")\
            .toTable("spotify_catalog.silver.DimArtist")

##DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimTrack/checkpoint")\
    .option("schemaEvaluationMode","addNewColumn")\
    .load("abfss://bronze@azstoragespotify.dfs.core.windows.net/DimTrack")
    

In [0]:
display(df_track, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimTrack/checkpoint")

In [0]:
df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
                                            .when(col('duration_sec')<300,"Medium")\
                                            .otherwise("High"))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

df_track = reusable().dropColumns(df_track,['_rescued_data'])


In [0]:
display(df_track, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/DimTrack/checkpoint")

In [0]:
df_track.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimTrack/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@azstoragespotify.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotify_catalog.silver.DimTrack")

##DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimDate/checkpoint")\
    .option("schemaEvaluationMode","addNewColumn")\
    .load("abfss://bronze@azstoragespotify.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])

df_date.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/DimDate/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@azstoragespotify.dfs.core.windows.net/DimDate/data")\
            .toTable("spotify_catalog.silver.DimDate")

##FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/FactStream/checkpoint")\
    .option("schemaEvaluationMode","addNewColumn")\
    .load("abfss://bronze@azstoragespotify.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact, checkpointLocation= "abfss://silver@azstoragespotify.dfs.core.windows.net/FactStream/checkpoint")

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@azstoragespotify.dfs.core.windows.net/FactStream/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@azstoragespotify.dfs.core.windows.net/FactStream/data")\
            .toTable("spotify_catalog.silver.FactStreamspotify_catalog.silver.factstream")

